Notebook for data exploration

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
data = pd.read_csv('NOVAIMS_projectData_B2C_202526.csv')

## 2. Data types and missing values

In [ ]:
data.info()

In [ ]:
data.isna().sum()

## 3. Variable Distributions

In [ ]:
# Correcting data types
data["Dairy"] = pd.to_numeric(
    data["Dairy"].astype(str).str.replace(",000", "", regex=False).str.strip(),
    errors="coerce"
)

data["ID_Store_last"] = data["ID_Store_last"].astype("category")

In [ ]:
data.describe(include='number')

In [ ]:
# Select numeric columns
numeric_cols = data.select_dtypes(include=["number"]).columns.tolist()

# Plot distributions
n_cols = 3
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
axes = axes.flatten()

for ax, col in zip(axes, numeric_cols):
    sns.histplot(data[col].dropna(), kde=True, ax=ax, bins=30)
    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel("Count")

for ax in axes[len(numeric_cols):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
data.describe(include='object')

In [ ]:
# Categorical variable distributions (bar plots)
cat_cols = data.select_dtypes(include=["object", "category"]).columns.tolist()

n_cols = 3
n_rows = (len(cat_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.5 * n_rows))
axes = axes.flatten()

for ax, col in zip(axes, cat_cols):
    # Plot counts (sorted by frequency)
    vc = data[col].value_counts(dropna=False)
    sns.barplot(x=vc.head(15).values, y=vc.head(15).index.astype(str), ax=ax, orient="h")
    ax.set_title(col)
    ax.set_xlabel("Count")
    ax.set_ylabel("")

for ax in axes[len(cat_cols):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Box and whiskers plot for product categories
product_cols = [
    'Beer', 'Bottled_Water', 'Bread', 'Meat', 'Dairy', 'Fresh_Foods',
    'Frozen_Foods', 'Fruit_Beverages', 'Pastry', 'Sodas', 'Toiletries', 'Veggies'
]

# Filter to only include columns that exist in the dataset
product_cols = [col for col in product_cols if col in data.columns]

# Create boxplot
plt.figure(figsize=(14, 6))
data[product_cols].boxplot(vert=True, patch_artist=True)
plt.xticks(rotation=45, ha='right')
plt.ylabel('Value')
plt.title('Box and Whiskers Plot for Product Categories')
plt.tight_layout()
plt.show()

## 4. Deep Dives

In [ ]:
lon_non_missing = data["Longevity_months"].dropna()

# Impossible values
pct_below_0 = (lon_non_missing < 0).mean() * 100
# Older than the store itself
pct_above_96 = (lon_non_missing > 96).mean() * 100

print(f"Percentage below 0 (non-missing): {pct_below_0:.2f}%")
print(f"Percentage above 96 (non-missing): {pct_above_96:.2f}%")

In [ ]:
# Client density by geographic location
fig, ax = plt.subplots(figsize=(10,10), subplot_kw={'projection': ccrs.PlateCarree()})
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.add_feature(cfeature.LAND, facecolor='lightgray')
ax.add_feature(cfeature.OCEAN, facecolor='lightblue')

hb = ax.hexbin(data['Longitude'], data['Latitude'], gridsize=50, 
               cmap='YlOrRd', mincnt=1, transform=ccrs.PlateCarree())
plt.colorbar(hb, ax=ax, label='Client Count')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Client Density by Geographic Location')
ax.gridlines(draw_labels=True)
plt.show()

In [ ]:
valid = (~data["Latitude"].isna()) & (~data["Longitude"].isna())
in_band = data["Latitude"].between(-5, 5) & data["Longitude"].between(-5, 5)
pct_in_band = (in_band[valid].mean() * 100) if valid.any() else 0.0

print(f"Percentage of valid lat-lon pairs between -5 and 5: {pct_in_band:.2f}%")

In [ ]:
pct_zero = (data["Returns"] == 0).mean() * 100
pct_below_zero = (data["Returns"] < 0).mean() * 100

print(f"Percentage of 0s (non-missing): {pct_zero:.2f}%")
print(f"Percentage below 0 (non-missing): {pct_below_zero:.2f}%")

In [ ]:
gender_lower = data["Gender"].astype(str).str.lower().str.strip()
valid_genders = gender_lower.isin(["female", "f", "m", "mm", "male"])
pct_valid = valid_genders.mean() * 100

print(f"Percentage of valid gender values: {pct_valid:.2f}%")

In [ ]:
# Distribution of Flagged variable
flagged_counts = data["Flaged"].value_counts(dropna=False)
flagged_pct = data["Flaged"].value_counts(dropna=False, normalize=True) * 100

print(flagged_pct)